In [2]:
import numpy as np

np.random.seed(42)

MIN_GREEN = 15.0
MAX_GREEN = 60.0
CYCLE_LENGTH = 120.0
NUM_PHASES = 4

TRAFFIC_VOLUMES = np.array([45.0, 30.0, 15.0, 25.0])

def calculate_intersection_delay(green_times):
    normalized_green = (green_times / np.sum(green_times)) * CYCLE_LENGTH
    delay_per_phase = (TRAFFIC_VOLUMES ** 2) / (normalized_green + 1e-5)
    return np.sum(delay_per_phase) * 10.0

def enforce_traffic_constraints(green_times):
    return np.clip(green_times, MIN_GREEN, MAX_GREEN)

num_particles = 20
dimensions = NUM_PHASES
max_iterations = 40
w = 0.5
c1 = 1.5
c2 = 1.5

X = np.random.uniform(MIN_GREEN, MAX_GREEN, (num_particles, dimensions))
for i in range(num_particles):
    X[i] = enforce_traffic_constraints(X[i])

V = np.random.uniform(-2, 2, (num_particles, dimensions))

p_best = X.copy()
p_best_fitness = np.array([calculate_intersection_delay(p) for p in X])

g_best_idx = np.argmin(p_best_fitness)
g_best = p_best[g_best_idx].copy()
g_best_fitness = p_best_fitness[g_best_idx]

initial_best_plan = g_best.copy()
initial_best_delay = g_best_fitness

for iteration in range(max_iterations):
    for i in range(num_particles):
        r1 = np.random.rand(dimensions)
        r2 = np.random.rand(dimensions)

        V[i] = (w * V[i]) + (c1 * r1 * (p_best[i] - X[i])) + (c2 * r2 * (g_best - X[i]))
        X[i] = X[i] + V[i]
        X[i] = enforce_traffic_constraints(X[i])

        current_delay = calculate_intersection_delay(X[i])

        if current_delay < p_best_fitness[i]:
            p_best[i] = X[i].copy()
            p_best_fitness[i] = current_delay

            if current_delay < g_best_fitness:
                g_best = X[i].copy()
                g_best_fitness = current_delay

final_optimized_plan = (g_best / np.sum(g_best)) * CYCLE_LENGTH
initial_normalized_plan = (initial_best_plan / np.sum(initial_best_plan)) * CYCLE_LENGTH
improvement_percentage = ((initial_best_delay - g_best_fitness) / initial_best_delay) * 100

print("=" * 65)
print("TRAFFIC SIGNAL CONTROLLER: PSO OPTIMIZATION RUN RESULTS")
print("=" * 65)
print(f"Traffic Volume Baseline (Phases 1-4): {TRAFFIC_VOLUMES} cars/min")
print("-" * 65)
print("INITIAL STATE (Unoptimized Control Plan):")
print(f" -> Assigned Phase Durations : {np.round(initial_normalized_plan, 2)} seconds")
print(f" -> Overall Intersection Delay: {initial_best_delay:.2f} cost units")
print("-" * 65)
print("FINAL STATE (Optimized Swarm Control Plan):")
print(f" -> Assigned Phase Durations : {np.round(final_optimized_plan, 2)} seconds")
print(f" -> Overall Intersection Delay: {g_best_fitness:.2f} cost units")
print("-" * 65)
print(f"EFFICIENCY GAIN: Delay reduced by {improvement_percentage:.2f}%!")
print("=" * 65)


TRAFFIC SIGNAL CONTROLLER: PSO OPTIMIZATION RUN RESULTS
Traffic Volume Baseline (Phases 1-4): [45. 30. 15. 25.] cars/min
-----------------------------------------------------------------
INITIAL STATE (Unoptimized Control Plan):
 -> Assigned Phase Durations : [50.99 23.87 22.53 22.6 ] seconds
 -> Overall Intersection Delay: 1150.52 cost units
-----------------------------------------------------------------
FINAL STATE (Optimized Swarm Control Plan):
 -> Assigned Phase Durations : [46.96 31.3  15.65 26.09] seconds
 -> Overall Intersection Delay: 1102.08 cost units
-----------------------------------------------------------------
EFFICIENCY GAIN: Delay reduced by 4.21%!
